### Step 0 17102003Hieu@

#### Using python to create database (sql server)

In [1]:
import requests
import pandas as pd
import pyodbc
import time
from datetime import datetime

def connect_to_sql_server():
    server = 'DESKTOP-5D9KSIC\\SQLEXPRESS01'
    db_name = 'master'
    database = 'Healthcare_provide'

    conn = pyodbc.connect('DRIVER={ODBC Driver 18 for SQL Server};'
                        'SERVER=' + server + ';'
                        'DATABASE=' + db_name + ';'
                        'Trusted_Connection=yes;'
                        'Encrypt=yes;'
                        'TrustServerCertificate=yes; ')
    conn.autocommit = True
    cursor = conn.cursor()
    
    try:
        cursor.execute(f"IF NOT EXISTS (SELECT * FROM sys.databases WHERE name = '{database}') CREATE DATABASE {database}")
        conn.commit()
    except pyodbc.ProgrammingError as e:
        print("Error creating database:", e)
    conn.close()  # Đóng kết nối cũ với master
    
    conn = pyodbc.connect('DRIVER={ODBC Driver 18 for SQL Server};'
                          'SERVER=' + server + ';'
                          'DATABASE=' + database + ';'
                          'Trusted_Connection=yes;'
                          'Encrypt=yes;'
                          'TrustServerCertificate=yes; ')
    return conn

### Step 1

#### Read file csv and import file to database by using python 

##### Read file csv (8 file)

In [2]:
path_1 = 'cities.csv'
path_2 = 'departments.csv'
path_3 = 'diagnoses.csv'
path_4 = 'insurance.csv'
path_5 = 'patients.csv'
path_6 = 'procedures.csv'
path_7 = 'providers.csv'
path_8 = 'visits.csv' # Fact table

In [3]:
df = pd.read_csv(path_8)

In [4]:
df.head(5)

,Date of Visit,Patient ID,Provider ID,Department ID,Diagnosis ID,Procedure ID,Insurance ID,Service Type,Treatment Cost,Medication Cost,Follow-Up Visit Date,Patient Satisfaction Score,Referral Source,Emergency Visit,Payment Status,Discharge Date,Admitted Date,Room Type,Insurance Coverage,Room Charges(daily rate)
0,1/1/2024,4928,1,2,1,2,1,Outpatient,841,21,1/9/2024,7,Self-Referral,No,Paid,NaN,NaN,NaN,603.4,0
1,1/1/2024,1083,4,1,4,2,2,Inpatient,535,27,NaN,8,Emergency,No,Paid,NaN,NaN,Semi-Private Room,414.4,30
2,1/1/2024,4534,4,2,1,2,2,Inpatient,422,70,NaN,7,Self-Referral,No,Paid,NaN,NaN,Semi-Private Room,365.4,30
3,1/1/2024,4504,4,1,4,4,3,Outpatient,811,136,1/20/2024,5,Self-Referral,No,Paid,NaN,NaN,NaN,NaN,0
4,1/1/2024,331,4,2,1,2,3,Outpatient,682,131,1/30/2024,5,Physician Referral,No,Paid,NaN,NaN,NaN,NaN,0


##### Table visits (fact table)

##### Check data type from table visits

In [5]:
df.dtypes

Date of Visit                  object
Patient ID                      int64
Provider ID                     int64
Department ID                   int64
Diagnosis ID                    int64
Procedure ID                    int64
Insurance ID                    int64
Service Type                   object
Treatment Cost                  int64
Medication Cost                 int64
Follow-Up Visit Date           object
Patient Satisfaction Score      int64
Referral Source                object
Emergency Visit                object
Payment Status                 object
Discharge Date                 object
Admitted Date                  object
Room Type                      object
Insurance Coverage            float64
Room Charges(daily rate)        int64
dtype: object

##### Format dtype object to datetime 

In [6]:
# Date of Visit, Follow-Up Visit Date, Discharge Date, Admitted Date - type Object => convert to type datetime
# Sửa theo data type trong python, và chỉnh datetime trong sql thành datetime2 mới import được
df[['Date of Visit', 'Follow-Up Visit Date', 'Discharge Date', 'Admitted Date']] = df[['Date of Visit', 'Follow-Up Visit Date', 'Discharge Date', 'Admitted Date']].apply(pd.to_datetime, errors='coerce')

##### Check info data

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Date of Visit               5000 non-null   datetime64[ns]
 1   Patient ID                  5000 non-null   int64         
 2   Provider ID                 5000 non-null   int64         
 3   Department ID               5000 non-null   int64         
 4   Diagnosis ID                5000 non-null   int64         
 5   Procedure ID                5000 non-null   int64         
 6   Insurance ID                5000 non-null   int64         
 7   Service Type                5000 non-null   object        
 8   Treatment Cost              5000 non-null   int64         
 9   Medication Cost             5000 non-null   int64         
 10  Follow-Up Visit Date        2507 non-null   datetime64[ns]
 11  Patient Satisfaction Score  5000 non-null   int64       

##### Các cột NULL của bảng Visits

In [8]:
df[['Follow-Up Visit Date', 'Discharge Date', 'Admitted Date', 'Room Type', 'Insurance Coverage']]

,Follow-Up Visit Date,Discharge Date,Admitted Date,Room Type,Insurance Coverage
0,2024-01-09,NaT,NaT,NaN,603.4
1,NaT,NaT,NaT,Semi-Private Room,414.4
2,NaT,NaT,NaT,Semi-Private Room,365.4
3,2024-01-20,NaT,NaT,NaN,NaN
4,2024-01-30,NaT,NaT,NaN,NaN
...,...,...,...,...,...
4995,NaT,2025-05-15,2025-05-14,General Ward,557.2
4996,NaT,2025-05-18,2025-05-14,Semi-Private Room,426.3
4997,2025-06-01,NaT,NaT,NaN,602.7
4998,NaT,2025-05-18,2025-05-14,Semi-Private Room,562.1


##### Check data NULL trong bảng visit

In [9]:
# Follow-Up Visit Date NULL gần 50% 
# Discharge Date và Admitted Date NULL bằng nhau => hơn 50% 
# Room Type NULL khoảng 50% 
# Insurance Coverage NULL 0,02%

df[['Follow-Up Visit Date', 'Discharge Date', 'Admitted Date', 'Room Type', 'Insurance Coverage']].isnull().sum()

Follow-Up Visit Date    2493
Discharge Date          3768
Admitted Date           3768
Room Type               2507
Insurance Coverage       119
dtype: int64

##### Fill NULL bảng Visits

In [10]:
df = df.fillna({
    'Follow-Up Visit Date': '',
    'Discharge Date': '',
    'Admitted Date': '',
    'Room Type': '',
    'Insurance Coverage': 0, # fill giá tiền bảo hiểm bằng 0
})


##### Check duplicate

In [11]:
df[df.duplicated(keep=False)]

,Date of Visit,Patient ID,Provider ID,Department ID,Diagnosis ID,Procedure ID,Insurance ID,Service Type,Treatment Cost,Medication Cost,Follow-Up Visit Date,Patient Satisfaction Score,Referral Source,Emergency Visit,Payment Status,Discharge Date,Admitted Date,Room Type,Insurance Coverage,Room Charges(daily rate)


### Step 2

#### Connect dataframe (CSV) to database

In [12]:
# Your connection parameters
server = 'DESKTOP-5D9KSIC\\SQLEXPRESS01'
db_name = 'master'
database = 'Healthcare_provide'

# Establish the connection
conn = pyodbc.connect('DRIVER={ODBC Driver 18 for SQL Server};'
                      'SERVER=' + server + ';'
                      'DATABASE=' + database + ';'
                      'Trusted_Connection=yes;'
                      'Encrypt=yes;'
                      'TrustServerCertificate=yes; ')

In [ ]:
def save_data_to_sql(conn, df):
    """
    Lưu dữ liệu từ DataFrame vào bảng SQL Server.
    Args:
        conn: Kết nối tới SQL Server.
        df: DataFrame chứa dữ liệu cần lưu.
    """
    cursor = conn.cursor()

    # Lấy danh sách các cột từ DataFrame và đảm bảo rằng các cột có khoảng trắng được bao trong dấu []
    columns = ', '.join([f"[{col}]" for col in df.columns])  # Add square brackets around column names
    placeholders = ', '.join(['?'] * len(df.columns))  # Tạo các placeholder cho VALUES (?, ?, ?)

    # Tạo câu truy vấn INSERT động
    insert_query = f"INSERT INTO Providers ({columns}) VALUES ({placeholders})"

    # Chuyển đổi DataFrame thành danh sách tuple
    rows = [tuple(row) for row in df.to_numpy()]

    # Thực thi truy vấn với executemany
    cursor.executemany(insert_query, rows)
    conn.commit()

save_data_to_sql(conn, df)

#### Sau khi import vào database thì Dùng python thao tác data qua database


In [13]:
Query = """ 
select 
V.[Patient ID],
V.[Provider ID],
V.[Department ID],
V.[Diagnosis ID],
V.[Procedure ID],
V.[Insurance ID],
[Date of Visit],
[Follow-Up Visit Date],
[Service Type],
[Treatment Cost],
[Medication Cost],
[Patient Satisfaction Score],
[Referral Source],
[Emergency Visit],
[Payment Status],
[Discharge Date],
[Admitted Date],
[Room Type],
[Insurance Coverage],
[Room Charges(daily rate)],
[Patient Name],
[Gender] as Gender_Patient,
[Age] as Age_patient,
[Race] as Race_patient,
Diagnosis
from dbo.Visit as V
Left join Patients as Pa
ON V.[Patient ID] = Pa.[Patient ID]
Left join dbo.Diagnoses as Di
On V.[Diagnosis ID] = Di.[Diagnosis ID]
"""

In [14]:
df = pd.read_sql(Query, conn)

C:\Users\hchb1\AppData\Local\Temp\ipykernel_4360\446648414.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(Query, conn)


In [15]:
df['Room Type'].value_counts()

Room Type
                     2507
General Ward          876
Semi-Private Room     822
Private Room          795
Name: count, dtype: int64

In [16]:
df.shape

(5000, 25)

##### Check Data lỗi

In [17]:
df.dtypes

Patient ID                             int64
Provider ID                            int64
Department ID                          int64
Diagnosis ID                           int64
Procedure ID                           int64
Insurance ID                           int64
Date of Visit                 datetime64[ns]
Follow-Up Visit Date                  object
Service Type                          object
Treatment Cost                         int64
Medication Cost                        int64
Patient Satisfaction Score             int64
Referral Source                       object
Emergency Visit                       object
Payment Status                        object
Discharge Date                        object
Admitted Date                         object
Room Type                             object
Insurance Coverage                   float64
Room Charges(daily rate)               int64
Patient Name                          object
Gender_Patient                        object
Age_patien

##### Format lại datetime của các cột date

In [18]:
# Date of Visit, Follow-Up Visit Date, Discharge Date, Admitted Date - type Object => convert to type datetime
df[['Date of Visit', 'Follow-Up Visit Date', 'Discharge Date', 'Admitted Date']] = df[['Date of Visit', 'Follow-Up Visit Date', 'Discharge Date', 'Admitted Date']].apply(pd.to_datetime, errors='coerce')

In [19]:
# Ngày khám lớn hơn ngày tái khám
data_error = df[df['Date of Visit'] > df['Follow-Up Visit Date']]

In [20]:
data_error.shape

(326, 25)

In [21]:
data_error.head(5)

,Patient ID,Provider ID,Department ID,Diagnosis ID,Procedure ID,Insurance ID,Date of Visit,Follow-Up Visit Date,Service Type,Treatment Cost,...,Discharge Date,Admitted Date,Room Type,Insurance Coverage,Room Charges(daily rate),Patient Name,Gender_Patient,Age_patient,Race_patient,Diagnosis
34,35,4,4,4,5,1,2025-01-04,2024-10-28,Outpatient,645,...,NaT,NaT,,550.9,0,Taylor Harris,Male,36,Hispanic,Hypertension
88,89,5,4,3,5,1,2025-01-04,2024-11-22,Outpatient,701,...,NaT,NaT,,541.1,0,Taylor Thomas,Female,61,Other,Fracture
119,120,4,4,3,5,1,2025-01-01,2024-11-11,Inpatient,777,...,NaT,NaT,,604.1,0,Lian Chen,Male,78,Asian,Fracture
128,129,1,4,4,5,1,2025-01-01,2024-11-30,Outpatient,406,...,NaT,NaT,,311.5,0,Sanjay Patel,Female,20,Asian,Hypertension
132,133,1,1,2,3,3,2025-01-02,2024-12-11,Emergency,424,...,NaT,NaT,,329.7,0,Barbara Smith,Female,24,White,Asthma


##### Loại bỏ data lỗi

In [22]:
df_final = df[~(df['Date of Visit'] > df['Follow-Up Visit Date'])]

In [23]:
df_final.head(5)

,Patient ID,Provider ID,Department ID,Diagnosis ID,Procedure ID,Insurance ID,Date of Visit,Follow-Up Visit Date,Service Type,Treatment Cost,...,Discharge Date,Admitted Date,Room Type,Insurance Coverage,Room Charges(daily rate),Patient Name,Gender_Patient,Age_patient,Race_patient,Diagnosis
0,1,1,4,3,2,2,2025-04-09,NaT,Outpatient,127,...,2025-04-14,2025-04-09,Semi-Private Room,179.2,30,Morgan Thompson,Male,18,Hispanic,Fracture
1,2,5,2,2,4,1,2024-06-23,NaT,Emergency,624,...,NaT,NaT,General Ward,563.5,10,Avery Anderson,Male,19,Hispanic,Asthma
2,3,5,3,2,4,2,2025-02-07,NaT,Emergency,301,...,2025-02-16,2025-02-07,Semi-Private Room,295.4,30,Aisha Khan,Female,20,Asian,Asthma
3,4,3,1,4,1,3,2024-09-15,2024-09-23,Outpatient,234,...,NaT,NaT,,285.6,0,Adanna Eze,Male,20,Black,Hypertension
4,5,5,1,2,2,3,2024-04-15,2024-05-13,Outpatient,621,...,NaT,NaT,,453.6,0,Haruto Chen,Male,20,Asian,Asthma


In [24]:
df_final.shape

(4674, 25)

In [25]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4674 entries, 0 to 4999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Patient ID                  4674 non-null   int64         
 1   Provider ID                 4674 non-null   int64         
 2   Department ID               4674 non-null   int64         
 3   Diagnosis ID                4674 non-null   int64         
 4   Procedure ID                4674 non-null   int64         
 5   Insurance ID                4674 non-null   int64         
 6   Date of Visit               4674 non-null   datetime64[ns]
 7   Follow-Up Visit Date        2181 non-null   datetime64[ns]
 8   Service Type                4674 non-null   object        
 9   Treatment Cost              4674 non-null   int64         
 10  Medication Cost             4674 non-null   int64         
 11  Patient Satisfaction Score  4674 non-null   int64         
 1

### Step tranform, tính toán ra các cột mới

In [26]:
df_final['Length of Stay'] = (df_final['Discharge Date'] - df_final['Admitted Date']).dt.days

C:\Users\hchb1\AppData\Local\Temp\ipykernel_4360\3905541692.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['Length of Stay'] = (df_final['Discharge Date'] - df_final['Admitted Date']).dt.days


In [27]:
df_final['Length of Stay'] = df_final['Length of Stay'].fillna(1)

C:\Users\hchb1\AppData\Local\Temp\ipykernel_4360\2470646856.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['Length of Stay'] = df_final['Length of Stay'].fillna(1)


In [29]:
df_final.dtypes

Patient ID                             int64
Provider ID                            int64
Department ID                          int64
Diagnosis ID                           int64
Procedure ID                           int64
Insurance ID                           int64
Date of Visit                 datetime64[ns]
Follow-Up Visit Date          datetime64[ns]
Service Type                          object
Treatment Cost                         int64
Medication Cost                        int64
Patient Satisfaction Score             int64
Referral Source                       object
Emergency Visit                       object
Payment Status                        object
Discharge Date                datetime64[ns]
Admitted Date                 datetime64[ns]
Room Type                             object
Insurance Coverage                   float64
Room Charges(daily rate)               int64
Patient Name                          object
Gender_Patient                        object
Age_patien

In [30]:
# Tính tổng số chi phí bệnh nhân phải trả
df_final['Total_cost_patients'] = df_final['Treatment Cost'] + df_final['Medication Cost'] + (df_final['Room Charges(daily rate)'] * df_final['Length of Stay'])

C:\Users\hchb1\AppData\Local\Temp\ipykernel_4360\3843711014.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['Total_cost_patients'] = df_final['Treatment Cost'] + df_final['Medication Cost'] + (df_final['Room Charges(daily rate)'] * df_final['Length of Stay'])


In [31]:
# Tính số tiền bệnh nhân phải trả = revenue
df_final['Amount_patient_paid'] = df_final['Total_cost_patients'] - df_final['Insurance Coverage'] 

C:\Users\hchb1\AppData\Local\Temp\ipykernel_4360\2732735049.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['Amount_patient_paid'] = df_final['Total_cost_patients'] - df_final['Insurance Coverage']


In [32]:
# Phân loại bệnh nhân thuộc khám bệnh hoặc khám bệnh xong nhập viện
df_final['Categories_Patient'] = df_final['Admitted Date'].apply(
    lambda x: 'patient_admitted_discharged' if pd.notna(x) else 'patient_examine'
)

C:\Users\hchb1\AppData\Local\Temp\ipykernel_4360\2747042399.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['Categories_Patient'] = df_final['Admitted Date'].apply(


In [33]:
df_final

,Patient ID,Provider ID,Department ID,Diagnosis ID,Procedure ID,Insurance ID,Date of Visit,Follow-Up Visit Date,Service Type,Treatment Cost,...,Room Charges(daily rate),Patient Name,Gender_Patient,Age_patient,Race_patient,Diagnosis,Length of Stay,Total_cost_patients,Amount_patient_paid,Categories_Patient
0,1,1,4,3,2,2,2025-04-09,NaT,Outpatient,127,...,30,Morgan Thompson,Male,18,Hispanic,Fracture,5.0,376.0,196.8,patient_admitted_discharged
1,2,5,2,2,4,1,2024-06-23,NaT,Emergency,624,...,10,Avery Anderson,Male,19,Hispanic,Asthma,1.0,805.0,241.5,patient_examine
2,3,5,3,2,4,2,2025-02-07,NaT,Emergency,301,...,30,Aisha Khan,Female,20,Asian,Asthma,9.0,662.0,366.6,patient_admitted_discharged
3,4,3,1,4,1,3,2024-09-15,2024-09-23,Outpatient,234,...,0,Adanna Eze,Male,20,Black,Hypertension,1.0,408.0,122.4,patient_examine
4,5,5,1,2,2,3,2024-04-15,2024-05-13,Outpatient,621,...,0,Haruto Chen,Male,20,Asian,Asthma,1.0,648.0,194.4,patient_examine
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4969,1,3,2,4,2,2025-01-21,NaT,Outpatient,841,...,30,Linda Jones,Male,77,White,Asthma,3.0,1109.0,374.7,patient_admitted_discharged
4996,4970,2,3,4,5,3,2024-08-11,NaT,Inpatient,422,...,10,Kwame Diallo,Male,77,Black,Hypertension,1.0,559.0,167.7,patient_examine
4997,4971,5,4,2,5,2,2024-04-07,NaT,Emergency,286,...,50,Linda Williams,Male,78,White,Asthma,1.0,380.0,114.0,patient_examine
4998,4972,2,3,3,2,3,2025-03-09,2025-04-02,Outpatient,554,...,0,Hina Chen,Female,78,Asian,Fracture,1.0,688.0,206.4,patient_examine


#### Outcome khi xử lý 

In [36]:
df_final[['Patient ID', 'Treatment Cost', 'Medication Cost', 'Room Charges(daily rate)', 'Length of Stay','Insurance Coverage', 'Total_cost_patients', 'Amount_patient_paid', 'Categories_Patient']].head(10)

,Patient ID,Treatment Cost,Medication Cost,Room Charges(daily rate),Length of Stay,Insurance Coverage,Total_cost_patients,Amount_patient_paid,Categories_Patient
0,1,127,99,30,5.0,179.2,376.0,196.8,patient_admitted_discharged
1,2,624,171,10,1.0,563.5,805.0,241.5,patient_examine
2,3,301,91,30,9.0,295.4,662.0,366.6,patient_admitted_discharged
3,4,234,174,0,1.0,285.6,408.0,122.4,patient_examine
4,5,621,27,0,1.0,453.6,648.0,194.4,patient_examine
5,6,314,26,0,1.0,238.0,340.0,102.0,patient_examine
6,7,541,96,10,1.0,452.9,647.0,194.1,patient_admitted_discharged
7,8,256,117,30,3.0,282.1,463.0,180.9,patient_admitted_discharged
8,9,577,35,0,1.0,428.4,612.0,183.6,patient_examine
9,10,886,85,10,1.0,686.7,981.0,294.3,patient_examine


In [38]:
df_final.shape

(4674, 28)

In [81]:
df_final.dtypes

Patient ID                             int64
Provider ID                            int64
Department ID                          int64
Diagnosis ID                           int64
Procedure ID                           int64
Insurance ID                           int64
Date of Visit                 datetime64[ns]
Follow-Up Visit Date          datetime64[ns]
Service Type                          object
Treatment Cost                         int64
Medication Cost                        int64
Patient Satisfaction Score             int64
Referral Source                       object
Emergency Visit                       object
Payment Status                        object
Discharge Date                datetime64[ns]
Admitted Date                 datetime64[ns]
Room Type                             object
Insurance Coverage                   float64
Room Charges(daily rate)               int64
Patient Name                          object
Gender_Patient                        object
Age_patien

###  Step 3 (final)

#### Import dataframe from Python to sql server database 

In [42]:
# Test using sqlalchemy
from sqlalchemy import create_engine, inspect
server = 'DESKTOP-5D9KSIC\\SQLEXPRESS01' 
database = 'Healthcare_provide'

connection_string = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;TrustServerCertificate=yes;"
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={connection_string}")

table_name = 'Visits_remove_data_error'

inspector = inspect(engine)
existing_tables = inspector.get_table_names()

if table_name in existing_tables:
    print(f"Table '{table_name}' have existed in '{database}'.")
else:
    try:
        df_final.to_sql(table_name, engine, if_exists='fail', index=False)
        print(f"Data has been inserted in '{table_name}' successfully!")
    except Exception as e:
        print(f"Data can't be inserted: {e}")

Data has been inserted in 'Visits_remove_data_error' successfully!
